# 15-bob. 15-BOB. ETIKA, MAXFIYLIK VA HUQUQIY ASOSLAR
## Amaliy misol: tushuntirish yashirin yorliqni fosh etadi

[![Colab'da ochish](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mrmsi/si-kiberxavfsizlik/blob/main/15-bob/15-1_tushuntirish-yashirin-yorliqni-fosh-etadi.ipynb)

*"Sun'iy intellektni kiberxavfsizlikda qo'llash" darsligi, 15-bob.*

Quyidagi dasturda fishing URL manzillarini aniqlovchi model "eski" ma'lumotlar to'plamida o'qitiladi. Bu to'plamda bir yashirin muammo bor: ma'lumotlar yig'ilgan davrda fishing saytlarning deyarli hammasi HTTPS shifrlashsiz ishlagan. Bugungi kunda esa bepul sertifikatlar paydo bo'lgani sababli fishing saytlarining ko'pchiligi ham HTTPS'dan foydalanadi. Model avval umumiy, so'ngra mahalliy tushuntirish orqali tahlil qilinadi va bugungi fishing saytlarida sinab ko'riladi. Mahalliy tushuntirishda har bir belgining hissasi logistik regressiya og'irligini belgining o'rtacha qiymatdan og'ishiga ko'paytirish orqali hisoblanadi. Dastur amaliy mashg'ulotning A15 mavzusiga mos keladi.

**Ishga tushirish:** yuqoridagi menyudan *Runtime → Run all* (*Ishga tushirish → Hammasini bajarish*) ni tanlang yoki kataklarni birma-bir Shift+Enter bilan bajaring. Barcha ma'lumotlar dasturning o'zida yaratiladi, qo'shimcha fayl yuklash shart emas.

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance

rng = np.random.default_rng(6)
belgilar = ["uzunlik", "nuqtalar", "raqamlar", "kalit_soz", "ip_manzil", "https_yoq"]

def urllar(n, fishing, eski_toplam):
    """URL belgilari (sun'iy ma'lumot). Eski to'plamda fishing saytlarning deyarli
    hammasi https'siz bo'lgan - bu to'plamni yig'ish jarayonining "tasodifiy izi"."""
    d = pd.DataFrame({
        "uzunlik": rng.normal(75 if fishing else 45, 18, n),
        "nuqtalar": rng.poisson(3.5 if fishing else 2, n),
        "raqamlar": rng.poisson(6 if fishing else 2, n),
        "kalit_soz": rng.random(n) < (0.5 if fishing else 0.1),    # "login", "verify" kabi so'zlar
        "ip_manzil": rng.random(n) < (0.08 if fishing else 0.01),
    })
    https_ulush = (0.05 if eski_toplam else 0.85) if fishing else 0.95
    d["https_yoq"] = rng.random(n) >= https_ulush
    return d.astype(float)

oqitish = pd.concat([urllar(3000, False, True), urllar(3000, True, True)])
y = np.array([0] * 3000 + [1] * 3000)
model = LogisticRegression(max_iter=2000).fit(oqitish, y)

In [2]:
# 1. Umumiy tushuntirish: belgini aralashtirib yuborganda sifat qanchalik tushadi
muhim = permutation_importance(model, oqitish, y, n_repeats=5, random_state=0)
print("Belgilar muhimligi (umumiy tushuntirish):")
for i in muhim.importances_mean.argsort()[::-1]:
    print(f"  {belgilar[i]:<10} {muhim.importances_mean[i]:.3f}")

Belgilar muhimligi (umumiy tushuntirish):
  https_yoq  0.123
  raqamlar   0.089
  uzunlik    0.026
  nuqtalar   0.007
  kalit_soz  0.006
  ip_manzil  0.002


In [3]:
# 2. Mahalliy tushuntirish: bitta URL uchun har bir belgining hissasi
url = urllar(1, True, True).iloc[0]
hissa = model.coef_[0] * (url - oqitish.mean())
print("\nBitta shubhali URL uchun belgilar hissasi:")
for b in hissa.abs().sort_values(ascending=False).index:
    print(f"  {b:<10} qiymat={url[b]:6.1f}  hissa={hissa[b]:+.2f}")


Bitta shubhali URL uchun belgilar hissasi:
  https_yoq  qiymat=   1.0  hissa=+2.57
  raqamlar   qiymat=   6.0  hissa=+1.96
  ip_manzil  qiymat=   1.0  hissa=+1.59
  nuqtalar   qiymat=   0.0  hissa=-1.54
  kalit_soz  qiymat=   0.0  hissa=-0.54
  uzunlik    qiymat=  58.0  hissa=-0.16


In [4]:
# 3. Bugungi fishing saytlar: ularning ko'pchiligi endi https ishlatadi
eski = urllar(2000, True, True)
yangi = urllar(2000, True, False)
print(f"\nFishing saytlarni aniqlash: eski to'plamda {model.predict(eski).mean():.1%}, "
      f"bugungi saytlarda {model.predict(yangi).mean():.1%}")


Fishing saytlarni aniqlash: eski to'plamda 98.1%, bugungi saytlarda 77.5%


## Kitobdagi natija

Dasturni ishga tushirganingizda quyidagiga o'xshash natija chiqishi kerak (tasodifiy sonlar generatori urug'i qat'iy belgilangani uchun natija odatda aynan bir xil bo'ladi; faqat bajarilish vaqti va har safar yangi yaratiladigan tasodifiy kalitlar farq qilishi mumkin):

```
Belgilar muhimligi (umumiy tushuntirish):
  https_yoq  0.123
  raqamlar   0.089
  uzunlik    0.026
  nuqtalar   0.007
  kalit_soz  0.006
  ip_manzil  0.002

Bitta shubhali URL uchun belgilar hissasi:
  https_yoq  qiymat=   1.0  hissa=+2.57
  raqamlar   qiymat=   6.0  hissa=+1.96
  ip_manzil  qiymat=   1.0  hissa=+1.59
  nuqtalar   qiymat=   0.0  hissa=-1.54
  kalit_soz  qiymat=   0.0  hissa=-0.54
  uzunlik    qiymat=  58.0  hissa=-0.16

Fishing saytlarni aniqlash: eski to'plamda 98.1%, bugungi saytlarda 77.5%
```

## O'zingiz sinab ko'ring

Parametrlarni (ma'lumotlar hajmi, chegara qiymati, tasodifiy son urug'i) o'zgartirib, natija qanday o'zgarishini kuzating. Kitobdagi 15-bobning B darajadagi topshiriqlari aynan shu daftar asosida bajariladi.